# 08 · Fase 4: matriz de recomendación

Dibuja el mapa de calor de los efectos entre categorías $L_{jk}$ de la fase 2: **filas** = categoría afectada $j$, **columnas** = categoría en promoción $k$. Cada celda es el cambio porcentual en las ventas de $j$ cuando 10 puntos porcentuales adicionales del surtido de $k$ están en promoción. Se marcan con un asterisco los pares significativos tras la corrección de Benjamini-Hochberg (FDR 5%).

| | |
|---|---|
| **Entrada** | `resultados/tablas/fase2_cruzados.csv` (de `06`) |
| **Salida** | `resultados/tablas/matriz_recomendacion.csv` y `resultados/figuras/figura_matriz_recomendacion.png` (300 dpi) |

Este notebook no estima nada: solo ordena y dibuja lo que produjo el `06`.

In [ ]:
# ------------------------------------------------------------------------------------------------------------
# Este notebook construye la matriz de efectos entre categorías (fase 4) a partir de los resultados de la fase 2.
# ------------------------------------------------------------------------------------------------------------

# Librerías
# ------------------------------------------------------------------------------------------------------------
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from matplotlib.patches import Rectangle

In [ ]:
# Parámetros
# ------------------------------------------------------------------------------------------------------------
# Núcleo de categorías, en el orden de las columnas (D-28)
nucleo_exposicion = [
    "MEATS", "POULTRY", "DELI", "PRODUCE", "DAIRY",
    "BREAD/BAKERY", "BEVERAGES", "LIQUOR,WINE,BEER", "GROCERY I", "CLEANING",
]
categorias_con_programa = ["MEATS", "POULTRY", "DELI", "PRODUCE"]    # sus columnas son promociones fuera del programa

# Colores: azul = sinergia (efecto positivo), naranja = canibalización (efecto negativo), gris = cero
azul, naranja, gris_medio, tinta = "#2563EB", "#C2410C", "#F3F4F6", "#1A1A2E"

# Rutas
# ------------------------------------------------------------------------------------------------------------
ruta_tablas = os.path.join("resultados", "tablas")
ruta_figuras = os.path.join("resultados", "figuras")
os.makedirs(ruta_figuras, exist_ok=True)

## 1. Matriz de efectos

In [ ]:
# Leer los efectos cruzados de la fase 2
cruzados = pd.read_csv(os.path.join(ruta_tablas, "fase2_cruzados.csv"))
assert not cruzados.duplicated(["afectada_j", "en_promocion_k"]).any(), "hay pares repetidos"

# Ordenar las filas como en el núcleo (solo las categorías estimadas) y las columnas con todo el núcleo
filas = [c for c in nucleo_exposicion if c in set(cruzados["afectada_j"])]
matriz = cruzados.pivot(index="afectada_j", columns="en_promocion_k", values="L_10pp_pct").reindex(index=filas, columns=nucleo_exposicion)
signif = cruzados.pivot(index="afectada_j", columns="en_promocion_k", values="significativo_bh").reindex(index=filas, columns=nucleo_exposicion)

# Verificar: la diagonal (la propia categoría) debe estar vacía y el resto completo
for j in filas:
    assert pd.isna(matriz.loc[j, j]), f"la diagonal de {j} debe estar vacía"
assert matriz.notna().sum().sum() == len(cruzados), "faltan o sobran pares en la matriz"

matriz.round(2).to_csv(os.path.join(ruta_tablas, "matriz_recomendacion.csv"))
print(matriz.round(1).to_string())
print(f"\nPares significativos tras Benjamini-Hochberg: {int(signif.fillna(False).sum().sum())} de {len(cruzados)}")

## 2. Mapa de calor

In [ ]:
def dibujar_matriz(matriz: pd.DataFrame, signif: pd.DataFrame, archivo: str) -> None:
    """
    Esta función dibuja el mapa de calor de los efectos entre categorías y lo guarda en PNG a 300 dpi.

    :param matriz:      un DataFrame con filas = categoría afectada y columnas = categoría en promoción, con el efecto
                        de +10 puntos porcentuales en porcentaje (NaN en la diagonal).
    :param signif:      un DataFrame booleano del mismo tamaño que marca los pares significativos tras la corrección.
    :param archivo:     la ruta del PNG de salida.
    """

    # Escala simétrica alrededor de cero: un solo color por signo, más oscuro = efecto mayor
    tope = float(np.nanmax(np.abs(matriz.to_numpy())))
    mapa = LinearSegmentedColormap.from_list("div", [naranja, gris_medio, azul])
    norma = TwoSlopeNorm(vmin=-tope, vcenter=0, vmax=tope)

    fig, ax = plt.subplots(figsize=(14, 0.8 * len(matriz) + 2.6))
    ax.imshow(np.ma.masked_invalid(matriz.to_numpy()), cmap=mapa, norm=norma, aspect="auto")

    # Etiquetas de ejes; las columnas con programa semanal se aclaran con una nota
    etiquetas_col = [c + ("†" if c in categorias_con_programa else "") for c in matriz.columns]
    ax.set_xticks(range(matriz.shape[1]), etiquetas_col, rotation=35, ha="right", fontsize=14)
    ax.set_yticks(range(matriz.shape[0]), matriz.index, fontsize=14)
    ax.xaxis.tick_top()
    plt.setp(ax.get_xticklabels(), rotation=35, ha="left", rotation_mode="anchor")
    ax.set_xticks(np.arange(-0.5, matriz.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-0.5, matriz.shape[0], 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=2)
    ax.tick_params(which="both", length=0)
    for lado in ax.spines.values():
        lado.set_visible(False)

    # Valor en cada celda; asterisco y recuadro en los pares significativos; "propia" en la diagonal
    for i, j in enumerate(matriz.index):
        for k, c in enumerate(matriz.columns):
            valor = matriz.iat[i, k]
            if pd.isna(valor):
                ax.add_patch(Rectangle((k - 0.5, i - 0.5), 1, 1, facecolor="#E5E7EB", hatch="///", edgecolor="white", lw=0))
                continue
            marca = "*" if bool(signif.iat[i, k]) else ""
            fuerte = abs(valor) > 0.6 * tope
            ax.text(k, i, f"{valor:+.1f}%{marca}", ha="center", va="center", fontsize=14,
                    fontweight="bold" if marca else "normal", color="white" if fuerte else tinta)
            if marca:
                ax.add_patch(Rectangle((k - 0.5, i - 0.5), 1, 1, fill=False, edgecolor=tinta, lw=2.5))

    ax.set_ylabel("Categoría afectada", fontsize=14)
    ax.set_title("Efecto en las ventas de la categoría afectada (fila) cuando +10 puntos porcentuales\n"
                 "del surtido de la categoría de la columna están en promoción", fontsize=15, loc="left", pad=75, fontweight="bold")
    fig.text(0.01, -0.02,
             "Azul = sinergia, naranja = canibalización. * y recuadro = significativo tras Benjamini-Hochberg (FDR 5%). "
             "Sin marca = sin efecto detectable (no es efecto nulo).\n"
             "† Categoría con programa semanal fijo: la columna mide solo sus promociones fuera del programa. "
             "Diagonal = la propia categoría (ya está en las variables propias). Efectos asociados, no causales.",
             fontsize=11, color="#4B5563", va="bottom")
    fig.subplots_adjust(left=0.17, right=0.98, bottom=0.07, top=0.70)
    fig.savefig(archivo, dpi=300, bbox_inches="tight")
    plt.show()


dibujar_matriz(matriz, signif, os.path.join(ruta_figuras, "figura_matriz_recomendacion.png"))
print(f"Figura guardada en {ruta_figuras}")

print(f"\n>>>> Proceso finalizado <<<<")